# 02. Классические модели

**Цель:** выбрать набор признаков и сравнить классические модели: Ridge, KNN, Random Forest, LightGBM, XGBoost, ансамбль.

**Метрика:** RMSE между предсказанием и `log1p(SalePrice)`.

**Валидация:** все модели проверяются на одних и тех же 5 фолдах `KFold`, поэтому результаты сравнимы.

In [1]:
from pathlib import Path
import sys
import yaml

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, cross_val_predict
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

from src.data import drop_outliers, set_categorical_features
from src.experiments import RANDOM_STATE, cv, cv_rmse, load_features, run_study, tune_ridge
from src.features import add_features, clean_data, encode_quality
from src.preprocessing import build_preprocessor, build_tree_preprocessor

# 1. Подготовка данных

Порядок шагов:
1. `clean_data` — заполняет пропуски, которые означают «нет объекта» (нет гаража, подвала и т.д.);
2. `add_features` — добавляет `TotalSF` и `TotalBath`; делается после заполнения пропусков;
3. `drop_outliers` — удаляет два недостроенных дома с огромной площадью и низкой ценой.

Настоящие пропуски (`LotFrontage`, `Electrical`) закрывает препроцессор внутри пайплайна.

In [2]:
config_path = project_root / "config.yaml"
with config_path.open(encoding="utf-8") as file:
    config = yaml.safe_load(file)

train = pd.read_csv(project_root / config["data"]["train_path"])

train_data = clean_data(train)
train_data = add_features(train_data)
train_data = drop_outliers(train_data)

In [3]:
numeric_features, categorical_features = load_features(train_data)
X_train = train_data[numeric_features + categorical_features]
y_train = np.log1p(train_data["SalePrice"])

results = {}  # название модели -> CV RMSE, для итоговой таблицы

# 2. Эксперименты с признаками (на Ridge)

Сравниваем три набора признаков

In [4]:
def report_ridge(name, X, num_features, cat_features):
    """Подбирает alpha для Ridge, печатает результат и сохраняет его в results."""
    alpha, rmse, std = tune_ridge(X, y_train, num_features, cat_features)
    print(f"Best alpha: {alpha:.3f}")
    print(f"Best CV RMSE: {rmse:.4f} ± {std:.4f}")
    results[name] = rmse

## 2.1. Базовый набор

Все признаки исходных данных, кроме удалённых по итогам EDA (дублирующие и почти константные — список `drop` в `config.yaml`), плюс новые признаки `TotalSF` и `TotalBath`.

In [5]:
report_ridge("Ridge", X_train, numeric_features, categorical_features)

Best alpha: 17.783
Best CV RMSE: 0.1138 ± 0.0080


## 2.2. Базовый набор без составных площадей

Базовый набор минус `TotalBsmtSF`, `2ndFlrSF`, `BsmtFinSF1`, `BsmtFinSF2`, `BsmtUnfSF`, то есть площади, которые являются частями или суммами других (`TotalSF`, `TotalBsmtSF`).

In [6]:
extra = ["TotalBsmtSF", "2ndFlrSF", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF"]
numeric_without_extra = [c for c in numeric_features if c not in extra]

report_ridge("Ridge без составных площадей", X_train, numeric_without_extra, categorical_features)

Best alpha: 17.783
Best CV RMSE: 0.1145 ± 0.0077


## 2.3. Базовый набор с порядковым кодированием качества

Оценки качества (`Po < Fa < TA < Gd < Ex`, а также "нет объекта") заменяются числами 0–5 вместо one-hot.

In [7]:
train_q = encode_quality(train_data)
numeric_q, categorical_q = load_features(train_q)
X_train_q = train_q[numeric_q + categorical_q]

report_ridge("Ridge с кодированием качества", X_train_q, numeric_q, categorical_q)

Best alpha: 17.783
Best CV RMSE: 0.1142 ± 0.0066


## Вывод по признакам

Три набора дают практически одинаковое качество, поэтому дальше используем самый простой вариант — **базовый набор**.

Переменные `numeric_features`, `categorical_features` и `X_train` ниже относятся именно к нему.

# 3. Другие модели

Все модели ниже обучаются на базовом наборе (`X_train`) и проверяются на тех же фолдах `cv`.

## 3.1 KNN

In [8]:
model = Pipeline([
    ("preprocessing", build_preprocessor(numeric_features, categorical_features)),
    ("knn", KNeighborsRegressor()),
])

search = GridSearchCV(
    estimator=model,
    param_grid={"knn__n_neighbors": range(1, 100)},
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
search.fit(X_train, y_train)

print(f"Best n_neighbors: {search.best_params_['knn__n_neighbors']}")
print(f"Best CV RMSE: {-search.best_score_:.4f}")
results["KNN"] = -search.best_score_

Best n_neighbors: 9
Best CV RMSE: 0.1694


## 3.2. Random Forest

Гиперпараметры подбираем через optuna (`run_study`) и сравниваем два способа кодирования категорий:
- **one-hot** (`build_preprocessor`) — тот же препроцессор, что у линейных моделей;
- **порядковое** (`build_tree_preprocessor`) — каждая категория заменяется номером

In [9]:
def make_objective_rf(build_prep):
    """Возвращает objective для optuna с заданным препроцессором."""
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
            "max_depth": trial.suggest_int("max_depth", 2, 16),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "max_features": trial.suggest_float("max_features", 0.2, 1.0),
        }
        model = Pipeline([
            ("preprocessing", build_prep(numeric_features, categorical_features)),
            ("rf", RandomForestRegressor(**params, random_state=RANDOM_STATE, n_jobs=-1)),
        ])
        return cv_rmse(model, X_train, y_train).mean()
    return objective

In [10]:
study_rf_onehot = run_study(
    make_objective_rf(build_preprocessor), "RF one-hot", params_key="rf_onehot"
)
results["Random Forest, one-hot"] = study_rf_onehot.best_value

  0%|          | 0/50 [00:00<?, ?it/s]

Best CV accuracy: 0.1290
Best params: {'n_estimators': 488, 'max_depth': 16, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 0.35239604381809403}


In [11]:
study_rf_ordinal = run_study(
    make_objective_rf(build_tree_preprocessor), "RF ordinal", params_key="rf_ordinal"
)
results["Random Forest, ordinal"] = study_rf_ordinal.best_value

  0%|          | 0/50 [00:00<?, ?it/s]

Best CV accuracy: 0.1292
Best params: {'n_estimators': 717, 'max_depth': 14, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.31896183712661}


**Вывод.** Способ кодирования почти не влияет на качество леса. Лес заметно хуже Ridge, так как после логарифма цена зависит от признаков почти линейно

## 3.3. LightGBM

Бустинг умеет работать с категориальными признаками напрямую, поэтому им передаём тип `category`, без one-hot.

In [12]:
X_train_boost = set_categorical_features(X_train, categorical_features)

In [13]:
def objective_lgbm(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "num_leaves": trial.suggest_int("num_leaves", 2, 31),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 40),
    }
    model = LGBMRegressor(**params, random_state=RANDOM_STATE, verbosity=-1)
    return cv_rmse(model, X_train_boost, y_train).mean()


study_lgbm = run_study(objective_lgbm, "LGBM hyperparameters search", params_key="lgbm")
results["LightGBM"] = study_lgbm.best_value

  0%|          | 0/50 [00:00<?, ?it/s]

Best CV accuracy: 0.1184
Best params: {'n_estimators': 672, 'max_depth': 3, 'num_leaves': 3, 'learning_rate': 0.07023069831858063, 'min_child_samples': 9}


## 3.4. XGBoost

In [14]:
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
    }
    model = XGBRegressor(
        **params,
        random_state=RANDOM_STATE,
        enable_categorical=True,
    )
    return cv_rmse(model, X_train_boost, y_train, n_jobs=-1).mean()


study_xgb = run_study(objective_xgb, "XGB hyperparameters search", params_key="xgb")
results["XGBoost"] = study_xgb.best_value

  0%|          | 0/50 [00:00<?, ?it/s]

Best CV accuracy: 0.1174
Best params: {'n_estimators': 460, 'max_depth': 2, 'min_child_weight': 1.1956919686025032, 'learning_rate': 0.03520628940669924, 'subsample': 0.6118976830930017}


**Вывод по бустингам.** Оба бустинга лучше леса и близки к Ridge. При этом optuna выбрала очень мелкие деревья (глубина 2–3)

# 4. Ансамбль моделей

Попробуем смешать Ridge и два вида бустинга.

для каждой модели берём **предсказания на валидации** (`cross_val_predict` на тех же фолдах `cv`), то есть каждый дом предсказывает модель, которая его не видела. Параметры берём из подбора выше (`best_params` у исследований optuna).

In [16]:
alpha, _, _ = tune_ridge(X_train, y_train, numeric_features, categorical_features)

models = {
    "Ridge": (
        Pipeline([
            ("preprocessing", build_preprocessor(numeric_features, categorical_features)),
            ("ridge", Ridge(alpha=alpha)),
        ]),
        X_train,
    ),
    "LightGBM": (
        LGBMRegressor(**study_lgbm.best_params, random_state=RANDOM_STATE, verbosity=-1),
        X_train_boost,
    ),
    "XGBoost": (
        XGBRegressor(**study_xgb.best_params, random_state=RANDOM_STATE, enable_categorical=True),
        X_train_boost,
    ),
}

oof = {
    name: cross_val_predict(model, X, y_train, cv=cv, n_jobs=-1)
    for name, (model, X) in models.items()
}

In [17]:
def fold_rmse(pred):
    """Средний RMSE по фолдам."""
    return np.mean([
        np.sqrt(np.mean((pred[val_idx] - y_train.iloc[val_idx]) ** 2))
        for _, val_idx in cv.split(X_train)
    ])


ensembles = {
    "Ridge + LightGBM + XGBoost": np.mean([oof["Ridge"], oof["LightGBM"], oof["XGBoost"]], axis=0),
    "Ridge + LightGBM": np.mean([oof["Ridge"], oof["LightGBM"]], axis=0),
    "Ridge + XGBoost": np.mean([oof["Ridge"], oof["XGBoost"]], axis=0),
}

for name, pred in oof.items():
    print(f"{name}: {fold_rmse(pred):.4f}")
print()
for name, pred in ensembles.items():
    print(f"Ансамбль {name}: {fold_rmse(pred):.4f}")
    results[f"Ансамбль: {name}"] = fold_rmse(pred)

Ridge: 0.1138
LightGBM: 0.1184
XGBoost: 0.1174

Ансамбль Ridge + LightGBM + XGBoost: 0.1115
Ансамбль Ridge + LightGBM: 0.1117
Ансамбль Ridge + XGBoost: 0.1108


**Вывод.** Смесь моделей даёт лучше результат, чем одиночные модели.

# 5. Итоги

In [19]:
summary = pd.Series(results, name="CV RMSE").sort_values().round(4).to_frame()
summary

,CV RMSE
Ансамбль: Ridge + XGBoost,0.1108
Ансамбль: Ridge + LightGBM + XGBoost,0.1115
Ансамбль: Ridge + LightGBM,0.1117
Ridge,0.1138
Ridge с кодированием качества,0.1142
Ridge без составных площадей,0.1145
XGBoost,0.1174
LightGBM,0.1184
"Random Forest, one-hot",0.1290
"Random Forest, ordinal",0.1292


**Вывод.**
Лучшим является ансамбль: Ridge + XGBoost, с помощью него будем предсказывать на тестовой выборке